# Customer Churn - Credit Card
---

### CRISP-DM Methodology
This project follows the CRISP-DM (*Cross-Industry Standard Process for Data Mining*) framework applied to **Customer Retention & Churn Prediction**:
| **Stage** | **Objective** | **Methodological Execution** |
| :--- | :--- | :--- |
| **1. Business Understanding** | Mitigate revenue loss by identifying at-risk customers. | • **Target Definition**: Binary Classification (Churn: Yes/No).<br>• **KPIs**: Maximize **Lift** in retention campaigns & Revenue Saved vs. Cost. |
| **2. Data Understanding** | Detect patterns of friction and dissatisfaction. | • **EDA**: Distribution analysis (Detect Imbalance).<br>• **Hypothesis Testing**: Correlation Matrix & Independence Tests (Chi-Square). |
| **3. Data Preparation** | Construct a robust dataset for parametric modeling. | • **Scaling**: Standardization (Z-score) for coefficient comparability.<br>• **Encoding**: One-Hot Encoding for nominal variables.<br>• **Splitting**: Stratified Train/Test Split to preserve class ratio. |
| **4. Modeling** | Estimate Churn Probability | • **Algorithms**: Logistic Regression, SVM LinearSVC, KNN, Random Florest, XGBoost, LightGBM.<br>• **Inference**: Analyze **Odds Ratios** to determine feature elasticity. |
| **5. Evaluation** | Assess model reliability and financial impact. | • **Discrimination**: AUC-ROC & F1-Score (Threshold Tuning).<br>• **Calibration**: Probability Calibration Curve (Reliability Diagram). |
| **6. Deployment** | Integrate insights into the CRM lifecycle. | • **Deliverable**: "High-Risk" Customer List for Marketing Squad.<br>• **Artifact**: Serialize model (`joblib`) for batch inference. |

---

### Installs:

In [0]:
%%capture
%pip install -r '../requirements.txt'
# Command to restart the kernel and update the installed libraries
%restart_python

### Imports:

In [0]:
# ======================================================== #
# Data Analysis and Visualization                          #
# ======================================================== #
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew, mannwhitneyu, pointbiserialr, chi2_contingency
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

# ======================================================== #
# Data Modeling                                            #
# ======================================================== #
from sklearn.model_selection import train_test_split

In [0]:
# SRC/ Functions Utils:
import sys
sys.path.append('../src')
from visualization import *
from utils import *

### Load the data

In [0]:
df = pd.read_csv('../data/BankChurners.csv')

### Verify successful load with some randomly selected records


In [0]:
df.sample(9)

## 1 - Business Understanding
---

### General Problem Context
#### What is Churn Rate and why is this metric relevant?

Customer attrition is one of the main challenges faced by companies that operate with recurring products, financial services, and long-term customer relationships.

In this context, the **churn rate** represents the rate of cancellation or customer turnover over a given period, making it an essential metric for assessing the company’s ability to **retain its active customer base** and sustain growth efficiently.

Before proposing any analytical or predictive solution, it is important to understand that churn is not only an operational problem, but also a phenomenon directly associated with **revenue loss**, higher customer replacement costs, and a reduction in the value generated throughout the customer lifecycle.

> From a strategic perspective, monitoring churn makes it possible to identify signs of dissatisfaction, weaknesses in the customer relationship, and possible failures in value delivery, turning this metric into one of the main indicators of business sustainability.

---

### Metric Calculation
#### How is Churn Rate calculated?

The churn rate will be defined based on the proportion between the number of customers lost during a time interval and the number of customers existing at the beginning of that same period. 

##### Churn Rate Formula:

$$
\text{Churn Rate} = \frac{\text{Number of customers lost during the period}}{\text{Total number of customers at the beginning of the period}} \times 100
$$


This formulation makes it possible to measure, in percentage terms, the intensity of customer attrition, serving as a reference for historical analysis, time-based comparisons, and the definition of retention targets. 

---

### High Churn Impacts
#### Why is a high churn rate a problem?

Although the total elimination of churn is unlikely in real-world scenarios, persistently high levels of this rate tend to compromise the company’s financial efficiency and scalability.

Among the main impacts of a high churn rate, the following stand out:

- Reduced recurring revenue and lower financial predictability.
- Greater commercial effort required to replace the lost customer base.
- Increased pressure on customer acquisition strategies.
- Lower return on investments in marketing, service, and relationship management.
- Erosion of the perceived value of the offered solution.

> From an analytical standpoint, a high churn rate may indicate structural failures in customer experience, misalignment in the value proposition, low product adherence to audience needs, or competitive vulnerability in the market.

---

### Main Causes
#### Factors associated with customer attrition

Customer departure rarely occurs due to a single isolated reason. In general, churn is the result of an accumulation of friction points throughout the customer journey.

The most recurrent factors include:

1. **Low perceived value**
- Occurs when the customer does not perceive enough concrete benefits to justify remaining with the contracted service or product.

2. **Unsatisfactory experience**
- Service issues, complex processes, operational failures, or poor usability can weaken the relationship with the company.

3. **Competitive pressure**
- More attractive competitor offers, whether in price, benefits, or convenience, may encourage customer migration.

4. **Changes in customer needs**
- Changes in customer profile, financial situation, or usage demand may reduce the solution’s fit with the consumer’s current context.

> Identifying these causes is essential to guide analytical hypotheses, segmentation strategies, and future evidence-based retention actions.

---

### Project Challenge
#### Business problem

The bank manager identified growth in the number of customers who are abandoning the credit card service.

Given this scenario, the main objective of the project will be to transform historical data into **actionable intelligence**, making it possible to understand the factors associated with churn and anticipate customer attrition risk.

Stakeholders expect the proposed solution to be capable of:

1. **Analyzing historical data** to identify patterns and variables related to churn.
2. **Developing a machine learning model** to estimate the probability of customer attrition.
3. **Supporting strategic retention actions**, prioritizing customers with the highest cancellation propensity.

> From a business perspective, the project aims to reduce customer losses, improve the efficiency of retention campaigns, and support data-driven decisions in the context of active customer relationship management.

---

### Project KPIs
#### Key performance indicators

To evaluate the analytical and strategic success of the churn prediction project, the following indicators will be monitored:

1. **Churn Rate**
- Percentage of customers who stopped using the service over a given period.
- This is the main business metric associated with the problem. 

2. **Retention Rate**
- Percentage of customers maintained in the base over time.
- It allows the effectiveness of implemented retention actions to be evaluated.

3. **CAC versus Retention Cost**
- Relationship between the cost of acquiring new customers and the cost required to retain existing ones.
- This indicator helps measure the economic viability of preventive churn strategies.

4. **AUC-ROC**
- Measures the model’s discriminative ability to distinguish customers likely to churn from those with low attrition probability.
- It will be used as one of the main predictive performance metrics.

5. **Recall**
- Measures the proportion of churn customers correctly identified by the model.
- It is especially important in this context, because reducing **false negatives** means reducing the risk of losing customers who could have been impacted by retention actions.

> In churn problems, prioritizing metrics such as **Recall** and **AUC-ROC** is statistically coherent, since the goal of the model is not only to classify correctly, but mainly to identify in advance the customers with the highest risk of leaving.
---



## 2 - Data Understanding

---

### Dataset Overview
This dataset contains information from 10,000 bank customers, including demographic, financial, and relationship-related attributes such as age, salary, marital status, credit card limit, and card category.

> These variables provide the analytical foundation for investigating behavioral patterns associated with customer attrition and for supporting the construction of predictive models.

---

### Data File
- **Data file**: `BankChurners.csv`

---

### Target Variable
The dependent target variable is **`Attrition_Flag`**, a categorical feature with binary classes:

1. **`Existing Customer`**
- Represents customers who remained active, that is, non-churners.

2. **`Attrited Customer`**
- Represents customers who discontinued their relationship with the credit card service, that is, churners.

> Since this is a **binary classification** problem, the target variable will be used to distinguish customers who remain in the base from those who are more likely to leave.

---

### Data Source
- **Dataset collected from Kaggle**:
[https://www.kaggle.com/datasets/sakshigoyal7/credit-card-customers?sort=votes&select=BankChurners.csv](https://www.kaggle.com/datasets/sakshigoyal7/credit-card-customers?sort=votes&select=BankChurners.csv)

- **Original dataset reference**:
[https://leaps.analyttica.com/home](https://leaps.analyttica.com/home)


---


#### Dictionary of Dataset
---

The dataset is composed of variables of different natures, organized into categories covering **identification**, **demographic profile**, **behavior**, **relationship**, and **financial transactions**.

---

- `CLIENTNUM`: Customer identifier variable, representing a unique code assigned to each account holder.

- `Attrition_Flag`: **Target** variable related to customer status; if the account has been closed, the customer is classified as `Attrited Customer`, otherwise as `Existing Customer`.

> This variable defines the central **binary classification problem** of the analysis, serving as the target for all predictive modeling strategies.

---

- `Customer_Age`: Demographic variable that indicates the customer's **age** in years.

- `Gender`: Demographic variable that identifies the customer's **gender**, where `M` stands for male and `F` stands for female.

- `Dependent_count`: Demographic variable that represents the **number of dependents** linked to the customer.

- `Education_Level`: Demographic variable that describes the account holder's **educational qualification**, such as high school, college graduate, or other academic levels.

- `Marital_Status`: Demographic variable that indicates the customer's **marital condition**, such as married, single, divorced, or unknown.

- `Income_Category`: Demographic variable that represents the account holder's **annual income range**, grouped into categorical income bands.

- `Card_Category`: Product-related variable that identifies the **type of credit card** held by the customer, such as Blue, Silver, Gold, or Platinum.

---

- `Months_on_book`: Relationship variable that indicates the **length of the customer's relationship** with the bank, in months.

- `Total_Relationship_Count`: Relationship variable that represents the **total number of financial products** held by the customer.

---

- `Months_Inactive_12_mon`: Behavioral variable that indicates the **number of inactive months** recorded for the customer during the last 12 months.

- `Contacts_Count_12_mon`: Behavioral variable that records the **number of contacts** made by the customer with the institution during the last 12 months.

---

- `Credit_Limit`: Financial variable that represents the **credit limit** assigned to the customer's credit card.

- `Total_Revolving_Bal`: Financial variable that indicates the **total revolving balance** carried on the credit card.

- `Avg_Open_To_Buy`: Financial variable that measures the **available credit line for use**, calculated as the average open-to-buy amount over the last 12 months.

- `Avg_Utilization_Ratio`: Financial behavior variable that represents the **average credit card utilization ratio**.



---

- `Total_Amt_Chng_Q4_Q1`: Transactional variable that captures the **variation in transaction amount** between the first and the fourth quarter.

- `Total_Trans_Amt`: Transactional variable that represents the **total amount transacted** by the customer over the last 12 months.

- `Total_Trans_Ct`: Transactional variable that indicates the **total number of transactions** performed by the customer over the last 12 months.

- `Total_Ct_Chng_Q4_Q1`: Transactional variable that measures the **variation in transaction count** between the first and the fourth quarter.

---



#### Exploratory Data Analysis (EDA):
---

#### 2.1 - Univariate Analysis:
---

> Examines the behavior of **a single variable** in isolation to understand its distribution, central tendency, and dispersion (e.g., histograms and means), without seeking relationships with other data.

In [0]:
df.head()

##### Dropping Redundant Columns
---

- According to the dataset documentation made available on **Kaggle**, the following columns were explicitly recommended for removal:

`Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_1`

`Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_2`

> These columns correspond to outputs from a **Naive Bayes classifier** applied by a previous contributor and carry no intrinsic predictive value for the current analysis. Their presence in the dataset represents a form of **data leakage**, as they encode information derived from the target variable itself.

---

- The `CLIENTNUM` column will also be removed, as it serves exclusively as a **unique customer registration identifier** for the banking institution.

> Since it is a purely administrative key with no statistical relationship to customer behavior or churn patterns, retaining it would introduce noise without contributing any **predictive or analytical value** to the modeling process.

---

In [0]:
redundants_cols = [
  'Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_1', 'Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_2',
  'CLIENTNUM'
]
df = df.drop(columns = [*redundants_cols])

# Check size Dataset
df.shape

In [0]:
print(df.info())

##### Adjusting Variable Types According to Their Characteristics
---

- In this step, the **data types** of the variables will be adjusted to formats more appropriate to the nature of each column.

This process aims to optimize **memory usage**, improve **processing efficiency**, and ensure greater consistency during exploratory analyses and the application of statistical tests.

> Correctly typing variables is a fundamental practice in data preprocessing, as it ensures that mathematical operations, aggregations, and comparisons are executed in a semantically coherent manner, preventing silent errors that could compromise the quality of the analysis.

---


In [0]:
df = optimize_dtypes(df)

print(f'New dtypes of variables:')
df.info()

print(f'Visual sample:')
df.head()

In [0]:
df.isnull().sum()

In [0]:
df[df.duplicated(keep = False)]

In [0]:
data_describe = df.describe()
data_describe

In [0]:
data_describe.loc['min']

In [0]:
data_describe.loc['max']

In [0]:
data_describe.loc['mean']

In [0]:
data_describe.loc['std']

In [0]:
numericals_cols = [
    'customer_age', 'dependent_count', 'months_on_book',
    'total_relationship_count', 'months_inactive_12_mon',
    'contacts_count_12_mon', 'credit_limit', 'total_revolving_bal',
    'avg_open_to_buy', 'total_amt_chng_q4_q1', 'total_trans_amt',
    'total_trans_ct', 'total_ct_chng_q4_q1', 'avg_utilization_ratio'
]
GraphicsData(data = df[numericals_cols]).numerical_histograms()

In [0]:

GraphicsData(data = df[numericals_cols]).numerical_boxplots(showfliers = True, showmeans = True)

In [0]:
categorical_cols = [
    'gender', 'education_level', 'marital_status',
    'income_category', 'card_category'
]
GraphicsData(data=df[categorical_cols]).categorical_countplots(figsize_per_row = 10)

In [0]:
GraphicsData(data = df).plot_target_analysis(target_col = 'attrition_flag', colors = ['#1abc9c', '#ff6b6b'])

##### Key Observations:
---

##### Data Integrity and Structural Quality

- The dataset contains **10,127 records** and **20 active variables**.
- No missing values were detected.
- No duplicate records were identified.
- Most variables are **numerical**, accounting for approximately **70%** of the dataset, while the remaining **30%** are **categorical**.

> The dataset shows strong structural integrity. There is no need for imputation or deduplication steps, which allows the analysis to move directly into statistical exploration and modeling with fewer preprocessing interventions.

> The dataset contains just over **10 thousand records**, which characterizes it as a medium-sized dataset. Although this is sufficient for exploratory analysis and initial modeling, this volume may impose limitations on the generalization of more complex models, depending on the quality of the variables and the imbalance of the target variable.
---

##### Age
---

- Average age of **46 years**

> The average age of customers is **46 years**, suggesting a relatively mature customer base. This profile may be associated with specific expectations regarding financial products and services, a hypothesis that should be validated throughout the analysis.
---

##### Dependents
---

- Average of **2 dependents**

> On average, customers have **2 dependents**, which contributes to the characterization of the dataset's family profile.
---

##### Relationship length (Tenure)
---

- Tenure of **35 months**

> The average relationship length with the bank is **35 months** (approximately 3 years), which suggests an already established relationship. Even so, the association between this variable and churn should be analyzed carefully.
---

##### Number of contracted services
---

- Average of **4 products/services**

> On average, customers hold approximately **4 products/services** with the bank. This variable may be relevant for evaluating the depth of the relationship and its possible association with customer attrition. In general, customers with a higher number of active services tend to be more loyal.
---

##### Marital status of customers
---

- The majority are **married**

> Most customers are **married**, representing about **46%** of the sample. During the analysis, it will be assessed whether marital status has any influence on customers' final behavior regarding the continuity of contracted services.
---

##### Education level
---

- Most customers have completed higher education

> A relevant share of customers have **completed higher education**, corresponding to approximately **30%** of the dataset. Historically, this variable often helps provide a better understanding of customers' demographic profiles. It also helps indicate their level of expectations regarding the contracted services.
---

##### Annual income of customers
---

- Annual income below **$40,000**

> Most customers report an annual income below **$40,000**. During the analysis, it will be assessed whether income has a direct impact on customer loyalty.
---

##### Cards and utilization
---

- Low credit card limit utilization

> On average, customers use around **27%** of their available credit limit.

- Basic cards are the majority

> The **Blue** card is the most common category, representing approximately **93%** of customers.
---

##### Customer gender
---

- Women are the majority

> The distribution between male and female customers is relatively balanced, with a slight predominance of female customers. The purpose of analyzing this variable is to verify whether there is any direct impact on the choice of contracted services and on customer loyalty regarding the services used.
---

##### Outliers
---

- Low number of outliers

> The **IQR** method was initially adopted to identify potential **outliers** in the numerical variables. Some columns contain values that may be statistically classified as outliers. The `Credit_Limit` variable, for example, has approximately **9.72%** of observations outside the limits defined by the IQR.

> These values should be evaluated in greater depth to determine whether they represent the natural distribution of the variable or possible inconsistencies in the data.

> At first, however, there is no indication of incorrect records or discrepant values that appear to be errors.
---

##### Churn rate
---

- The overall churn rate is approximately **16.1%**

> This percentage represents a relevant level of customer attrition and reinforces the need to investigate, in greater depth, the factors associated with churn.
---



#### 2.2 Bi-Variate Analysis:
---

Explores the statistical relationship between **two variables simultaneously**, aiming to identify associations, correlations, patterns of separation, or structural dependencies (e.g., scatterplot of `Income` vs. `Debt`, boxplot of `total_trans_count` vs. `churn`).

---

##### Methodological Note:
---

- Prior to conducting the bivariate analysis, the dataset will be partitioned into **Train** and **Test** sets.  
The primary objective is to prevent **Data Leakage**, ensuring that all statistical insights, outlier treatments, transformation decisions, and feature engineering strategies are derived exclusively from the training data.

> This approach preserves the integrity of the validation process and guarantees that performance metrics reflect true generalization capacity rather than information contamination.

- The `stratify` parameter will be applied within the `train_test_split` procedure.  
Given that churn prediction is a **binary classification problem**, maintaining the original **class proportion** across both subsets is statistically mandatory.

> Stratified sampling preserves the prior probability distribution of the target variable, avoiding distortions in class balance that could bias model training, threshold calibration, and performance evaluation metrics (e.g., Recall, Precision, ROC-AUC).

---


##### Adjusting the target variable
---

I will adjust the target column, `Attrition_Flag`. Since this is a categorical variable with binary classification:

---

- **Existing Customer (Non-churner)**: will receive the **value 0**.

  These are customers who still use the credit card services provided by the bank.

---

- **Attrited Customer (Churner)**: will receive the **value 1**.

---

> This approach will facilitate the calculation of correlation statistics between variables, since the target variable will already be properly encoded.

---


In [0]:
map_churn = {
    'Existing Customer': 0,
    'Attrited Customer': 1
}
df['attrition_flag'] = df['attrition_flag'].map(map_churn).astype('int8')
df = df.rename(columns={'attrition_flag': 'churn'})

In [0]:
train_set, test_set = train_test_split(
    df, 
    test_size = 0.2, 
    stratify = df['churn'], 
    random_state = 33
)

In [0]:
# Checking the proportions of the target variable
print(f'Shape of training: {train_set.shape}')
print(f'Shape of test: {test_set.shape}')

print('\n--- Churn Rate (Stratify Validation) ---')
print(f'Original: {df['churn'].mean():.2%}')
print(f'Train:    {train_set['churn'].mean():.2%}')
print(f'Test:    {test_set['churn'].mean():.2%}')

##### Checking the correlations between the variables

In [0]:
numerical_cols = [
    'customer_age', 'dependent_count', 'months_on_book',
    'total_relationship_count', 'months_inactive_12_mon',
    'contacts_count_12_mon', 'credit_limit', 'total_revolving_bal',
    'avg_open_to_buy', 'total_amt_chng_q4_q1', 'total_trans_amt',
    'total_trans_ct', 'total_ct_chng_q4_q1', 'avg_utilization_ratio', 'churn'
]

In [0]:
train_set[numerical_cols].corr()['churn'].abs().sort_values(ascending = False)

In [0]:
GraphicsData(data = train_set[numerical_cols]).correlation_heatmap()

##### Key Observations on the correlation between numerical variables and the `churn` variable
---
**Considering that a customer who churns is assigned the value 1 and a customer who does not churn is assigned the value 0:**

---
- **total_trans_ct**

> This is the variable with the strongest correlation with **churn**, showing a negative correlation of **-0.37**. This indicates that the lower the number of transactions made in the last 12 months, the greater the likelihood that the customer will cancel the bank's services.
---
- **total_ct_chng_q4_q1**

> This variable showed a correlation of **-0.29**. This column represents the change in the number of transactions between the fourth quarter (Q4) of the previous year and the first quarter (Q1). The lower this index, the greater the likelihood that the customer will cancel the bank's services.
---
- **total_revolving_bal**

> Customers with a lower **revolving balance** are more likely to become **inactive**, while customers with a higher revolving balance tend to continue using the bank's credit card services. It is worth noting that, even with a **higher revolving balance**, which may lead to potential future debt, these customers still tend to remain **active** within the institution.
---
- **contacts_count_12_mon**

> The **number of contacts** made in the last 12 months showed a **positive correlation**, indicating that a higher number of contacts is statistically associated with the rate of **inactive customers**.
---
- **avg_utilization_ratio**

> **Credit card limit utilization** shows a **negative correlation** with customers who have abandoned the service, meaning that **the lower the card usage**, the greater the likelihood that the customer will become **inactive**. On the other hand, the higher the credit limit utilization, the greater the likelihood that the customer will continue using the service.
---
- **total_relationship_count**

> The **number of services** shows a negative correlation with customers who have stopped using the credit card; **the lower the number of services**, the greater the likelihood that the customer will become **inactive**.
---
- **Number of inactive months**

> The **number of inactive months** shows a **positive correlation** with inactive customers, since the higher this number, the greater the likelihood that the customer will become inactive.
---
- **total_amt_chng_q4_q1**

> This variable shows a **negative correlation** with inactive customers; the lower the total value of changes between the fourth quarter (Q4) of the previous year and the first quarter (Q1), the greater the likelihood that the customer will become inactive.
---
- **avg_open_to_buy** and **credit_limit**

> These variables show a **perfect correlation**, indicating that both are providing the same information to machine learning models.
---
- **Other variables**

> The **remaining variables** do not show a highly relevant correlation with the **churn** rate in this dataset.
---

> At this stage, it is possible to conclude that the **number of transactions** made by customers in recent months has a **very strong and meaningful correlation** with the problem under analysis, namely customer discontinuation of the bank's services. The **two variables** most clearly associated with the target variable **churn** are precisely those that provide information about the **number of transactions performed by customers**. Based on these observed patterns, it is already possible to raise questions and formulate initial hypotheses.
---


##### Analyzing the numerical variables and their relationships with the target variable.

In [0]:
GraphicsData(data = train_set[numerical_cols]).numerical_histograms(hue = 'churn')

In [0]:
GraphicsData(data = train_set[numerical_cols]).numerical_boxplots(hue = 'churn', showmeans = True)

##### Statistical tests - of Numeric Variables
---

##### Mann-Whitney U Test
> I used the Mann-Whitney U test to verify the relationships between the native characteristics of this dataset and churn. **This choice is due to the high skewness (> 1) of the numerical variables**, which requires a non-parametric approach robust to outliers, where traditional tests (such as the t-test) failed.
---

In [0]:
EDATest(data = train_set).mannwhitney_u_test(audit_vars =  numerical_cols, target = 'churn')



##### Key Observations on numeric variables in relation to the `churn` variable
---
##### Number of Transactions

- *No-Churners*: **Mean of 68.67**

- *Churners*: **Mean of 44.81**

> In the **total_trans_ct** variable, it is possible to observe that most customers who stopped using the credit card recorded **fewer than 80 transactions in the last 12 months**. Customers with **95 transactions or more**, on the other hand, continued using the bank's services.

> In terms of averages, churners recorded a mean of **44.81** transactions in the last 12 months, compared with **68.67** for customers who remained active. The difference between these averages is greater than 20 points, and the statistical tests show that this variable has a strong correlation and high statistical significance **(p-value < 0.05)** in relation to churn.
---
##### Total Transaction Amount

- *No-Churners*: **Mean of 4655.87**

- *Churners*: **Mean of 3106.56**

> In the **total_trans_amt** variable, it is possible to observe that churners are more concentrated in the **lower transaction value ranges**. Most of these customers recorded **total transactions below \$2750**.

> In terms of averages, churners recorded a mean of **3106.56** in transactions over the last 12 months, compared with **4655.87** for customers who remained active. Statistical tests show that this variable has a strong correlation and high statistical significance **(p-value < 0.05)** in relation to churn.
---
##### Revolving Balance Amount

- *No-Churners*: **Mean of 1257.09**

- *Churners*: **Mean of 652.01**

> In the **total_revolving_bal** variable, it is possible to observe that the highest concentration of churners is found in the **lowest revolving balance values**. A significant portion of these customers have a **revolving balance below \$500**.

> Statistical tests show that this variable has high statistical significance and a strong correlation with churn.
---
##### Change in Number of Transactions

- *No-Churners*: **Mean of 0.74**

- *Churners*: **Mean of 0.56**

> In the **total_ct_chng_q4_q1** variable, it is possible to observe that most loyal customers showed an increase of at least **50%** in the number of transactions made between the fourth quarter (Q4) and the first quarter (Q1).

> This variable also showed high statistical significance and a strong correlation with churn.
---
##### Credit Card Utilization Rate

- *No-Churners*: **Mean of 0.30**

- *Churners*: **Mean of 0.16**

> In the **avg_utilization_ratio** variable, it is possible to observe that most churners **did not use their credit card limit** in the last 12 months.

> Loyal customers have an average utilization rate almost 2x higher than churners. This variable also shows high correlation and high statistical significance in relation to churn.
---

##### Customer Contacts with the Institution in the Last 12 Months

- *No-Churners*: **Mean of 2.36**

- *Churners*: **Mean of 2.95**

> In the **contacts_count_12_mon** variable, it is possible to observe that most customers who stopped using the credit card had a **number of contacts greater than or equal to 3**. This variable also shows high statistical significance and a strong correlation with churn.
---

##### Number of Services Contracted by the Customer

- *No-Churners*: **Mean of 3.92**

- *Churners*: **Mean of 3.30**

> Most churners maintained **3 services or fewer**. This variable showed strong statistical significance and a strong correlation with churn.
---

##### Customer Inactivity Months

- *No-Churners*: **Mean of 2.27**

- *Churners*: **Mean of 2.70**

> Overall, most customers recorded 3 months of inactivity or less. However, churners showed a slightly longer period of inactivity compared with loyal customers. This variable showed strong statistical significance and a strong correlation with churn.
---

##### Credit Limit and Available Credit

- **credit_limit**, **avg_open_to_buy**

> These variables showed strong statistical significance, but low correlation with churn. This effect is likely due to the nature of the data, since both variables are extremely skewed, which suggests that part of their statistical value in relation to Pearson correlation is being suppressed by the wide right-tailed distribution. In other words, higher values appear less frequently, while lower values are more concentrated on the left side of the distribution.

> Even so, from a statistical perspective, these variables do have a direct influence on churn and may still add important value to the models.
---
##### Other Variables

- **customer_age**, **dependent_count**, **months_on_book**

> Overall, these variables showed low statistical significance and low correlation with churn. Statistically, this suggests that these variables, on their own, do not have a direct relationship with churn. Customer **age**, **number of dependents**, and, somewhat surprisingly, **relationship length with the institution** do not directly affect customers' decision to remain with or discontinue the bank's services.
---


In [0]:
train_set.select_dtypes(include = ['category', 'int8']).columns

##### Analyzing the categorical variables and their relationships with the target variable.

In [0]:
categorical_cols = [
'gender', 'education_level', 'marital_status',
'income_category', 'card_category', 'churn'
]
GraphicsData(data = train_set[categorical_cols]).categorical_countplots(hue = 'churn')

In [0]:
GraphicsData(data = train_set[categorical_cols]).categorical_bar_percentages(hue = 'churn')

##### Categorical Features
> Chi Square Test
---

In [0]:
EDATest(train_set).chi_square_test(audit_vars = categorical_cols, target = 'churn')

##### Key Observations on categorical variables in relation to the `churn` variable
---

##### Education

- *Best category*: **College (15%)**
- *Worst category*: **Doctorate (22%)**

> **Education level** does not show a very strong relationship with the churn rate. In general, it would be expected that higher or lower levels of education could influence customers' decision to remain with or leave the bank's services. Even so, it is still possible to extract relevant observations from these data, as they may directly affect the institution's decision-making. The education level with the highest churn rate is **Doctorate**, at around **21.50%**.

> The **Chi-Square Test** showed that, statistically, this variable is not significant in relation to churn. Therefore, it is possible to conclude that, on its own, it does not have a direct impact on customers' decision-making.
---
##### Income

- *Best category*: **$60K - $80K (14%)**
- *Worst category*: **Unknown (17%)**

> **Customers' annual income** does not show a significant influence on the rate of customers who stopped using the credit card, since all income brackets follow a fairly similar distribution in relation to churn. Only customers in the **$60K - $80K** income range showed a churn rate of **14.10%**, which is slightly lower than the other income groups.

> The **Chi-Square Test** showed that, statistically, this variable is not significant in relation to churn. Therefore, it is possible to conclude that, on its own, it does not have a direct impact on customers' decision-making.
---

##### Credit card category

- *Best category*: **Silver (14%)**
- *Worst category*: **Platinum (26%)**

> The **credit card category** shows a relevant relationship with the rate of customers who stopped using the card. The **Gold** and **Platinum** categories recorded above-average cancellation rates compared with the other categories. However, these two categories represent a very small share of this dataset; together, they account for less than **2%** of the total sample. The **Silver** category has the lowest credit card cancellation rate, with a **churn rate of 14.4%**.

> The **Chi-Square Test** showed that, statistically, this variable is not significant in relation to churn, and it also pointed to a very small number of samples within these categories.
---

##### Gender

- *Best category*: **M (15%)**
- *Worst category*: **F (17%)**

> **Customer gender** showed a specific relationship with churn. Women recorded a higher credit card cancellation rate than men.

> Among the categorical variables, this was the only one that showed statistical relevance in the **Chi-Square Test**, with **(p-value < 0.05)**.
---
##### Marital status

- *Best category*: **Married (15%)**
- *Worst category*: **Unknown (18%)**

> **Marital status** also suggests a relationship with churn rates. **Married customers** show a **slightly lower** churn rate than **single customers**. However, this variable did not show statistical relevance in the **Chi-Square Test**, indicating that, on its own, it does not have a significant impact on customer behavior.
---

> At this stage, considering the statistical results and charts for these categorical variables, it is possible to conclude that they do not show strong enough relevance to solve the institution's churn problem on their own. Even so, some variables do show differences across their classes in relation to churn.



#### 2.3 Multi-Variate Analysis:
---

> Analyzes **three or more variables simultaneously** in order to uncover complex interaction effects, hidden dependency structures, and latent behavioral patterns that cannot be identified through univariate or bivariate approaches alone.
This stage is essential for understanding how combinations of demographic, behavioral, and financial variables jointly influence churn probability.

---

In [0]:
GraphicsData(train_set[numerical_cols]).scatterplots_vs_reference(
    x_reference = 'total_trans_ct',
    hue = 'churn',
    title = 'Scatterplot Of Numeric Variables x total_trans_ct Grouped By Churn Rate'
)


##### Key observations on numeric variables x `total_trans_ct` in relation to the `churn` variable
---

- The **total_trans_ct** variable shows a very strong correlation with the **churn** variable. For this reason, it was selected to assess its dispersion together with the other variables, grouped by the target variable **churn**.

> The combinations that provided the clearest separation between **Non-churners** and **Churners** were:

- **total_trans_ct** x **total_trans_amt**
- **total_trans_ct** x **total_ct_chng_q4_q1**
- **total_trans_ct** x **total_ct_chng_q4_q1**

> These variables are related to the number or total value of transactions, reinforcing the previous observations that transaction volume and total transaction value reflect, to some extent, customer behavior and may indicate whether the customer will continue using the credit card or discontinue its use.

> The variables **total_revolving_bal** and **avg_utilization_ratio**, together with **total_trans_ct**, showed a reasonable separation between the **Churn**


##### Creating News Features

In [0]:
# =========================================================
# A. Percentage/Ratio Features
# =========================================================


# avg_ticket
train_set['avg_ticket'] = (train_set['total_trans_amt'].astype('float64') / (train_set['total_trans_ct']).astype('float64')).astype('float32')

# inactive_per_tenure
train_set['inactive_per_tenure'] = ((train_set['months_on_book'] ).astype('float64') / (train_set['months_inactive_12_mon']).astype('float64')).astype('float32')

# credit_limit_per_tenure
#train_set['credit_limit_per_tenure'] = ((train_set['credit_limit'] ).astype('float64') / (train_set['months_on_book']).astype('float64')).astype('float32')


# =========================================================
# B. Interaction/Combination Features
# =========================================================

train_set['total_spending'] = (train_set['total_trans_amt'].astype('float64') + (train_set['total_revolving_bal'] ).astype('float64')).astype('float32')

# change_gap_amt_ct
train_set['change_gap_amt_ct'] = (train_set['total_amt_chng_q4_q1'].astype('float64') + (train_set['total_ct_chng_q4_q1']).astype('float64')).astype('float32')

# utilization_amont
train_set['utilization_amont'] = ((train_set['credit_limit']).astype('float64') * (train_set['avg_utilization_ratio'] ).astype('float64')).astype('float32')

# credit_to_limit
train_set['credit_revolving'] = ((train_set['credit_limit'] ).astype('float64') - (train_set['total_revolving_bal']).astype('float64')).astype('float32')

# =========================================================
# C. Binary Features / Flags
# =========================================================

# months_inactive_12_mon_0
train_set['months_inactive_12_mon_0'] = (train_set['months_inactive_12_mon'] == 0).astype('int8')

# contacts_count_12_mon_0
train_set['contacts_count_12_mon_0'] = (train_set['contacts_count_12_mon'] == 0).astype('int8')

# dependent_count_0
train_set['dependent_count_0'] = (train_set['dependent_count'] == 0).astype('int8')

# total_revolving_bal_0
train_set['total_revolving_bal_0'] = (train_set['total_revolving_bal'] == 0).astype('int8')

# total_amt_chng_q4_q1_0
train_set['total_amt_chng_q4_q1_0'] = (train_set['total_amt_chng_q4_q1'] == 0).astype('int8')

# total_ct_chng_q4_q1_0
train_set['total_ct_chng_q4_q1_0'] = (train_set['total_ct_chng_q4_q1'] == 0).astype('int8')

# avg_utilization_ratio_0
train_set['avg_utilization_ratio_0'] = (train_set['avg_utilization_ratio'] == 0).astype('int8')


# low_activity_low_value_flag
util_q = pd.qcut(
    train_set['avg_utilization_ratio'],
    q=4,
    labels=False,
    duplicates='drop'
)

revol_q = pd.qcut(
    train_set['total_revolving_bal'],
    q=4,
    labels=False,
    duplicates='drop'
)

amt_q = pd.qcut(
    train_set['total_trans_amt'],
    q=4,
    labels=False,
    duplicates='drop'
)

relation_q = pd.qcut(
    train_set['total_relationship_count'],
    q=4,
    labels=False,
    duplicates='drop'
)

trans_q = pd.qcut(
    train_set['total_trans_ct'],
    q=4,
    labels=False,
    duplicates='drop'
)

train_set['low_activity_low_value_flag'] = (
    (util_q == util_q.min()) &   
    (trans_q == trans_q.min()) &           
    (amt_q == amt_q.min()) &
    (revol_q == revol_q.min()) &
    (relation_q == relation_q.min())
).astype('int8')

# is_pos_graduate
train_set['is_pos_graduate'] = (
    (train_set['education_level']  == 'Doctorate') |   
    (train_set['education_level']  == 'Post-Graduate') 
).astype('int8')

# total_amt_q75
train_set['total_amt_q75'] = (
    (train_set['total_amt_chng_q4_q1']  >= 0.75)
).astype('int8')

#total_ct_q75
train_set['total_ct_q75'] = (
    (train_set['total_ct_chng_q4_q1']  >= 0.75)
).astype('int8')

# =======================================================
# D. Buckets / Ordinal Segmentation
# =========================================================
# trans_ct_bucket
train_set['trans_ct_bucket'] = pd.cut(
    train_set['total_trans_ct'],
    bins = [0, 25, 50, 75, 100, np.inf],
    labels = ['very_low_activity', 'low_activity', 'medium_activity', 'high_activity', 'very_high_activity'],
    include_lowest = True,
    ordered = True
)

# trans_ct_bucket
train_set['trans_amt_bucket'] = pd.cut(
    train_set['total_trans_amt'],
    bins = [0, 2500, 5000, 7500, 10000, np.inf],
    labels = ['very_low_activity', 'low_activity', 'medium_activity', 'high_activity', 'very_high_activity'],
    include_lowest = True,
    ordered = True
)

# tenure_bucket
train_set['tenure_bucket'] = pd.cut(
    train_set['months_on_book'],
    bins = [0, 12, 24, 48, np.inf],
    labels = ['new', 'developing', 'established', 'loyal'],
    include_lowest = True,
    ordered = True
)

num_cols = train_set.select_dtypes(include=[np.number]).columns
train_set[num_cols] = (
    train_set[num_cols]
    .replace([np.inf, -np.inf], 0)
)

##### Feature engineering for churn modeling
---

##### Overview

> An additional feature engineering layer was developed to enhance the model’s predictive power through the creation of variables derived from transactional behavior, relationship intensity, financial profile, and customer lifecycle stage.

> These new variables were structured into four main groups: **ratios and proportions**, **attribute interactions**, **binary flags**, and **ordinal segmentations**, with the goal of capturing richer signals than those available in the original variables.
---

##### Ratios

> The first block focused on proportional features designed to contextualize transaction volume, monetary activity, inactivity, and relationship depth. Instead of analyzing only absolute totals, metrics such as transactions per product, transaction value per relationship length, transaction count per dependent, value per contact, and inactivity per tenure were created, allowing customers with different structural profiles to be compared on a more standardized basis.

> These variables are especially useful in churn modeling because they help identify relative usage intensity rather than only raw consumption levels. For example, two customers may have the same total transaction value, but very different behavioral patterns when that value is adjusted for number of products, customer contacts, dependents, or relationship length.
---

##### Interactions

> The second block brought together interaction and combination features aimed at translating compound signals of friction, declining activity, and financial pressure. This group includes variables such as total spending combined with revolving balance, the sum of inactive months and number of contacts, the difference between changes in transaction amount and transaction count, utilization adjusted by inactivity, and the interaction between customer contacts and declining transaction activity.

> The rationale behind these features is to capture situations in which churn risk does not emerge from a single variable in isolation, but rather from the combination of multiple signals. From a business perspective, this helps represent patterns such as customers with low activity but high support demand, or customers showing recent deterioration in transactional behavior alongside signs of relationship friction.
---

##### Flags and buckets

> The third block was composed of binary flags used to highlight specific segments of greater analytical interest. These included variables identifying customers with high contact volume and low transactional activity, the presence of premium cards, and a composite indicator of low activity and low value, built from customers simultaneously positioned in the lower quartiles of utilization, transaction count, transaction value, revolving balance, and relationship depth.

> The fourth block introduced ordinal bucket variables with the objective of transforming continuous variables into more interpretable ranges. Buckets were created for number of transactions, transaction value, and relationship length, making it possible to segment customers by activity level and relationship maturity.
---



### 2.5 - EDA Conclusions
---
##### Number of transactions

> Based on the hypothesis tests, it is possible to conclude that the number of transactions made by customers using their credit cards has a significant impact on the churn rate. The distribution charts show that the number of transactions among customers who stopped using the institution’s credit card services is considerably lower. The average number of transactions for **non-churners** is nearly **69**, while **churners** average **45 transactions**.
---
##### Total transaction value

> Based on the hypothesis tests, it is possible to conclude that the total value of transactions made by customers using their credit cards has a significant impact on the churn rate. Most customers who stopped using the credit card services recorded a total transaction value **equal to or below US\$ 2,500**.
---
##### Credit card limit utilization rate

> The credit card limit utilization rate showed high significance in relation to the institution’s churn rate, and this was validated through the hypothesis tests. In this dataset, there is an **asymmetric distribution with a long right tail**, indicating that most credit limit utilization values are concentrated at the **lower levels**. This is an important point for the institution’s stakeholders to explore, since most customers use a **credit limit far below expectations**, which may represent a problem, considering that a large share of the bank’s **credit card profits** is associated with customer usage. In this regard, **non-churners** show an average limit utilization of **29%**, while **churners** show an average of **16%**.
---
##### Number of services maintained by the customer

> The number of services maintained by the customer on the card showed a significant impact on the institution’s churn rate, and this was also confirmed through the hypothesis tests.
---
##### Credit limit

> The **credit card limit** shows an **asymmetric distribution**, with most limits **below US\$ 5,000**. This is associated with the customer profile, since most customers have an annual income **below US\$ 40K**. Even so, the possibility of adopting a more flexible policy regarding credit limit increases may be assessed together with the institution’s stakeholders, taking customer default rates into account.
---
##### Card categories

> The **card categories** should be restructured with more flexible policies for the **Silver, Gold, and Platinum** tiers, as these three categories together represent **less than 7% of the total customer base**. The **Blue** category accounts for **93.1% of all customers**, indicating that most customers remain only in the entry-level card tier. This may not be a positive factor for the bank, considering that benefits and advantages tend to be more attractive in categories above Blue and, as a result, naturally support stronger customer loyalty and continued usage.
---
##### Contacts with the bank

> Churners showed a rate of **26% more contacts with the bank**, which reinforces the importance of understanding and addressing the needs of these customers.
---
##### Months of inactivity

> Most customers in this institution remained inactive for three months or less. However, churners showed **18% more months of inactivity** than non-churners who continued using the credit card services, indicating that customer inactivity is a critical issue that the bank should address and explore in potential solutions.
---
